# Stage 4 + Stage 5: Corpus Cleaning (`think-cleaned` -> `think-cleaned-1930s`, ported)

This notebook runs **after** `midtraining_ocroscope_filtering.ipynb` (Stage 3). It ports the
two cleaning passes originally prototyped against `jbduran/think-dataset` in
[`cleaning info/think_dataset_clean.ipynb`](cleaning%20info/think_dataset_clean.ipynb) and
[`cleaning info/think_dataset_clean_1930s.ipynb`](cleaning%20info/think_dataset_clean_1930s.ipynb)
onto our own pipeline's output (`zachnorton03/think-midtrain/stage3`), instead of reprocessing
`jbduran/think-dataset` from scratch.

**Why port instead of reference `cleaning info/` directly?** `cleaning info/` documents *how the
original `think-dataset-clean(-1930s)` repos were produced* -- it is reference material, not a
dependency. This notebook is self-contained: the reusable filtering logic (banned-term matcher,
footer stripper, structural/OCR cleanup) lives in [`scripts/`](scripts/) at the repo root, copied
and decoupled from the original HF-repo-specific orchestration so it has zero import-time
dependency on `cleaning info/`.

## What each stage does

| Stage | Source notebook | What it does |
|---|---|---|
| **4** | `think_dataset_clean.ipynb` | Boilerplate/OCR cleanup (Google Books, HathiTrust, library stamps, Project Gutenberg headers, hyphenation, front matter) + a GPT-2 token log-prior filter (p2.5-p97.5) that prunes statistical outliers (OCR garbage / repetitive boilerplate) missed by the regex pass. |
| **5** | `think_dataset_clean_1930s.ipynb` | Line-level footer/boilerplate stripping, then a **tiered** anachronism keyword filter: drop a document only on strong evidence of post-1930 content (one hard/tier-1 hit, or >=2 corroborating tier-2/3 hits). Avoids polysemy false positives ("compiler of this volume", bee "drone", birdsong "twitter"). |

## Data flow

```
zachnorton03/think-midtrain/stage3/data/*.parquet   (OCRoscope-filtered, from Stage 3)
        |
        v   Stage 4: structural clean + GPT-2 log-prior filter
zachnorton03/think-midtrain/stage4/data/*.parquet
zachnorton03/think-midtrain/stage4/_prior/           (cached GPT-2 log-prior table + thresholds)
        |
        v   Stage 5a: footer/boilerplate line strip
zachnorton03/think-midtrain/stage5/stripped/*.parquet
        |
        v   Stage 5b: tiered anachronism filter
zachnorton03/think-midtrain/stage5/data/*.parquet
zachnorton03/think-midtrain/stage5/_banned/          (cached banned-term list + tiers)
```

Unlike the original `think-dataset-clean(-1930s)` prototypes (single `text` column, one row per
book), every stage3+ shard in this pipeline carries **rich per-document metadata**
(`title_src`, `ocr_score_gen`, `ocroscope_ratio_segment`, `token_count_o200k_base_gen`,
`ia_identifier`, ...) alongside `text` in the *same* parquet file. Both stages here preserve that
metadata unchanged for every kept row and simply drop the row entirely for filtered-out documents
-- no separate metadata file or positional join is needed (stage3's shards are already
self-contained, unlike stage2's split text/metadata layout).

## Resumability

Both stages are resumable at shard granularity, matching Stage 2/Stage 3's convention:
- **Stage 4** skips any source shard whose Stage 4 output shard already exists on HF.
- **Stage 5** skips any source shard already present under `stage5/stripped/`, and separately
  skips any stripped shard already present under `stage5/data/` -- so a disconnect during the
  filter step never forces a re-strip, and vice versa.
- The GPT-2 log-prior table (Stage 4) and the banned-term list (Stage 5) are each built once,
  cached to HF, and downloaded on every rerun instead of being recomputed.
- A guarded **reset** cell is provided for each stage (default OFF) for the rare case you
  intentionally change a threshold after some shards already went out -- shard boundaries would
  otherwise be inconsistent between an old threshold and a new one.


## 0. Setup: scripts/, dependencies, HF connection

Pulls in the repo-root `scripts/` modules (added to `sys.path`), installs dependencies, and
connects to the same `zachnorton03/think-midtrain` HF dataset repo used by Stages 2-3. The HF
token comes from Colab Secrets (`HF_TOKEN_Z`), exactly as in the Stage 2/3 notebooks.

If running in Colab, upload/sync the repo-root `scripts/` folder into this Colab session first
(drag-and-drop, or mount Drive / clone the repo) so the import below resolves.


In [ ]:
%pip -q install -U datasets huggingface_hub pyarrow tqdm numpy transformers

import os, sys, json, re, time, tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import tqdm
from huggingface_hub import HfApi, hf_hub_download, list_repo_files
from google.colab import userdata

# --- Make the repo-root scripts/ package importable ------------------------
# Adjust SCRIPTS_DIR if you placed the repo somewhere else in this session
# (e.g. after `!git clone` or mounting Drive).
SCRIPTS_DIR = Path("scripts")
assert SCRIPTS_DIR.is_dir(), (
    f"Could not find '{SCRIPTS_DIR}/' in this session. Bring the repo-root scripts/ folder "
    "(build_list.py, filter_lib.py, footer_lib.py, structural_clean.py) into the working "
    "directory -- e.g. `!git clone <repo-url>` then `%cd think.nano-midtraining` -- and re-run."
)
sys.path.insert(0, str(SCRIPTS_DIR.resolve()))

import build_list
import filter_lib
import footer_lib
import structural_clean

print("Loaded scripts/:", [m.__name__ for m in (build_list, filter_lib, footer_lib, structural_clean)])

HF_TOKEN = userdata.get('HF_TOKEN_Z')
HF_REPO_ID = 'zachnorton03/think-midtrain'
api = HfApi(token=HF_TOKEN)

CHARS_PER_SHARD = 250_000_000  # keep the same shard size as upstream stages


def list_repo_paths():
    """Current file set in the dataset repo, re-listed each call so resume checks see
    checkpoints written earlier in THIS run too."""
    return set(list_repo_files(repo_id=HF_REPO_ID, repo_type='dataset', token=HF_TOKEN))


DL_MAX_RETRIES = 5
DL_ETAG_TIMEOUT = 30
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '30')


def robust_hf_download(path_in_repo, repo_id=None):
    """hf_hub_download with bounded per-attempt timeout + retry. A hung HF CDN node
    (the 'reconstructing file' bar frozen mid-stream) raises ReadTimeout instead of
    blocking forever; each retry re-resolves to a (usually) healthy node."""
    last_exc = None
    for attempt in range(1, DL_MAX_RETRIES + 1):
        try:
            return hf_hub_download(
                repo_id=repo_id or HF_REPO_ID, repo_type='dataset', filename=path_in_repo,
                token=HF_TOKEN, etag_timeout=DL_ETAG_TIMEOUT, force_download=(attempt > 1),
            )
        except Exception as exc:
            last_exc = exc
            print(f"  download retry {attempt}/{DL_MAX_RETRIES} for {path_in_repo}: {exc}")
    raise last_exc


def fmt_secs(s):
    m, s = divmod(int(s), 60)
    h, m = divmod(m, 60)
    return f"{h}h{m:02d}m" if h else f"{m}m{s:02d}s"


---
# Stage 4: Structural Cleanup + GPT-2 Log-Prior Filter

Port of `think_dataset_clean.ipynb`. Reads Stage 3 shards, applies `structural_clean` (Google
Books / HathiTrust / library-stamp / Project Gutenberg boilerplate removal, unicode/OCR-ligature
normalization, hyphenation rejoin, paragraph reflow, front-matter trim, then a length /
printable-ratio / OCR-artifact-density gate), then filters surviving documents on a GPT-2 mean
token log-prior band. Whole documents are kept or dropped -- the surviving text itself is the
*cleaned* text (boilerplate removed), not the raw Stage 3 text.

Design choices carried over unchanged from the original notebook:
- Whole books stay as single rows (no re-chunking).
- The post-1900 physics keyword filter is **skipped** -- this corpus intentionally keeps
  material through the 1930s (that filter is a Stage 5 job).
- Prior band: **p2.5-p97.5**, targeting ~5% prior-filter removal by construction.


In [ ]:
# === Stage 4 configuration ==================================================
SRC4_PREFIX = 'stage3'                 # input: OCRoscope-filtered corpus
DST4_PREFIX = 'stage4'                 # output: structurally-cleaned + prior-filtered corpus
DATA4_PREFIX = f'{DST4_PREFIX}/data'
PRIOR4_PREFIX = f'{DST4_PREFIX}/_prior'

# Structural gate thresholds -- override the script defaults here if needed.
structural_clean.MIN_CHARS_RAW = 500
structural_clean.MIN_CHARS_CLEAN = 500
structural_clean.MIN_PRINTABLE = 0.85
structural_clean.MAX_OCR_ARTIFACTS = 50

# Prior filter.
PRIOR_BAND = (2.5, 97.5)
SAMPLE_SHARDS = 12
SAMPLE_DOCS = 20_000
TOKENIZE_CHARS = 50_000
RANDOM_SEED = 42
FORCE_RECOMPUTE_PRIOR = False   # set True to rebuild the prior table from scratch

import random
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Stage 4: {HF_REPO_ID}/{SRC4_PREFIX} -> {HF_REPO_ID}/{DST4_PREFIX}")
print(f"Prior band: p{PRIOR_BAND[0]}-p{PRIOR_BAND[1]}")


## 4.0 (optional, guarded) Reset Stage 4 outputs

Only needed if you changed a structural/prior threshold after some Stage 4 shards were already
uploaded. Leave `CONFIRM_RESET_STAGE4 = False` for a normal resumable run.


In [ ]:
CONFIRM_RESET_STAGE4 = False
WIPE_STAGE4_PRIOR = False   # also wipe the cached prior table (forces a rebuild)

if CONFIRM_RESET_STAGE4:
    stale = [
        f for f in list_repo_paths()
        if f.startswith(f'{DATA4_PREFIX}/') or f == f'{DST4_PREFIX}/manifest.json'
        or (WIPE_STAGE4_PRIOR and f.startswith(f'{PRIOR4_PREFIX}/'))
    ]
    print(f"Deleting {len(stale)} file(s) under {DST4_PREFIX}/ ...")
    for f in stale:
        api.delete_file(path_in_repo=f, repo_id=HF_REPO_ID, repo_type='dataset',
                        commit_message=f'Reset Stage 4: remove {f}')
        print(f"  deleted {f}")
    print("Reset complete.")
else:
    print("Reset skipped (CONFIRM_RESET_STAGE4=False). Normal resumable run.")


## 4.1 Build or load the GPT-2 log-prior table

Ports `build_prior_from_sample()`: samples up to `SAMPLE_DOCS` structurally-cleaned documents
from `SAMPLE_SHARDS` random Stage 3 shards, tokenizes with GPT-2, and derives per-token
`log2(count / total)` priors plus the `[p2.5, p97.5]` band on each sampled document's mean prior.
Cached under `stage4/_prior/` on HF; a rerun loads the cache instead of resampling.


In [ ]:
from transformers import GPT2TokenizerFast
from collections import Counter

tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')

src4_shards = sorted(
    f for f in list_repo_paths()
    if f.startswith(f'{SRC4_PREFIX}/data/') and f.endswith('.parquet')
)
print(f"Found {len(src4_shards)} Stage 3 shard(s).")
assert src4_shards, "No Stage 3 shards found -- run midtraining_ocroscope_filtering.ipynb first."


def try_download_prior():
    try:
        prior_path = robust_hf_download(f'{PRIOR4_PREFIX}/log_priors.npy')
        thresh_path = robust_hf_download(f'{PRIOR4_PREFIX}/thresholds.json')
        log_priors = np.load(prior_path)
        with open(thresh_path) as f:
            thresholds = json.load(f)
        print("Loaded cached prior table + thresholds from HF.")
        return log_priors, thresholds
    except Exception:
        return None, None


def doc_mean_log_prior(text, log_priors):
    ids = tokenizer.encode(text[:TOKENIZE_CHARS], add_special_tokens=False)
    if not ids:
        return None
    ids = [i for i in ids if i < len(log_priors)]
    if not ids:
        return None
    return float(log_priors[ids].mean())


def build_prior_from_sample():
    rng = random.Random(RANDOM_SEED)
    sample_shards = src4_shards[:]
    rng.shuffle(sample_shards)
    sample_shards = sample_shards[:min(SAMPLE_SHARDS, len(sample_shards))]

    docs = []
    stage_estimates = Counter()
    raw_seen = 0

    print(f"Sampling up to {SAMPLE_DOCS:,} docs from {len(sample_shards)} shard(s) for priors...")
    for shard in tqdm.tqdm(sample_shards, desc="sample shards"):
        local = robust_hf_download(shard)
        pf = pq.ParquetFile(local)
        for rg_idx in range(pf.num_row_groups):
            table = pf.read_row_group(rg_idx, columns=['text'])
            for text in table.column('text').to_pylist():
                raw_seen += 1
                cleaned, reason = structural_clean.structural_clean(text)
                if cleaned is None:
                    stage_estimates[reason] += 1
                    continue
                docs.append(cleaned)
                if len(docs) >= SAMPLE_DOCS:
                    break
            if len(docs) >= SAMPLE_DOCS:
                break
        if len(docs) >= SAMPLE_DOCS:
            break

    if not docs:
        raise RuntimeError("No documents survived the structural sample; loosen thresholds.")

    counts = Counter()
    doc_token_ids = []
    total_tokens = 0
    for text in tqdm.tqdm(docs, desc="tokenizing sample"):
        ids = tokenizer.encode(text[:TOKENIZE_CHARS], add_special_tokens=False)
        if not ids:
            continue
        counts.update(ids)
        doc_token_ids.append(ids)
        total_tokens += len(ids)
    assert total_tokens > 0, "Tokenization sample produced zero tokens."

    vocab_size = tokenizer.vocab_size
    log_priors = np.full(vocab_size, np.log2(1.0 / total_tokens), dtype=np.float32)
    for tok_id, count in counts.items():
        if tok_id < vocab_size:
            log_priors[tok_id] = np.log2(count / total_tokens)

    means = np.array([float(log_priors[[i for i in ids if i < vocab_size]].mean()) for ids in doc_token_ids])
    lo, hi = np.percentile(means, PRIOR_BAND)
    thresholds = {
        'prior_band': list(PRIOR_BAND),
        'low': float(lo),
        'high': float(hi),
        'sample_raw_docs_seen': int(raw_seen),
        'sample_structural_kept': int(len(docs)),
        'sample_structural_removed': dict(stage_estimates),
        'sample_prior_docs': int(len(means)),
        'estimated_prior_removed_pct': float(100.0 * ((means < lo) | (means > hi)).mean()),
    }

    tmp = tempfile.mkdtemp()
    local_prior = os.path.join(tmp, 'log_priors.npy')
    local_thresh = os.path.join(tmp, 'thresholds.json')
    np.save(local_prior, log_priors)
    with open(local_thresh, 'w') as f:
        json.dump(thresholds, f, indent=2, sort_keys=True)
    api.upload_file(path_or_fileobj=local_prior, path_in_repo=f'{PRIOR4_PREFIX}/log_priors.npy',
                    repo_id=HF_REPO_ID, repo_type='dataset', commit_message='Stage 4 prior table')
    api.upload_file(path_or_fileobj=local_thresh, path_in_repo=f'{PRIOR4_PREFIX}/thresholds.json',
                    repo_id=HF_REPO_ID, repo_type='dataset', commit_message='Stage 4 prior thresholds')
    print("Sample-based prior estimates:")
    print(json.dumps(thresholds, indent=2))
    return log_priors, thresholds


if not FORCE_RECOMPUTE_PRIOR:
    LOG_PRIORS, PRIOR_THRESHOLDS = try_download_prior()
else:
    LOG_PRIORS, PRIOR_THRESHOLDS = None, None
if LOG_PRIORS is None:
    LOG_PRIORS, PRIOR_THRESHOLDS = build_prior_from_sample()

PRIOR_LOW, PRIOR_HIGH = PRIOR_THRESHOLDS['low'], PRIOR_THRESHOLDS['high']
print(f"Using mean log-prior band [{PRIOR_LOW:.4f}, {PRIOR_HIGH:.4f}]")


## 4.2 Resumable shard loop: clean, filter, upload

For every Stage 3 shard: structurally clean each doc, drop docs failing the structural gate or
the prior band, keep every metadata column unchanged for survivors (with `text` replaced by the
cleaned text), and upload a same-named output shard + stats JSON in one commit. A shard already
present under `stage4/data/` is skipped, so a disconnect costs at most the shard in flight.


In [ ]:
def process_one_row(row):
    """row: dict with at least 'text' + metadata. Returns (kept_row_or_None, reason)."""
    text = row.get('text')
    cleaned, reason = structural_clean.structural_clean(text)
    if cleaned is None:
        return None, reason

    mean_prior = doc_mean_log_prior(cleaned, LOG_PRIORS)
    if mean_prior is None:
        return None, 'empty_tokenization'
    if mean_prior < PRIOR_LOW:
        return None, 'prior_low'
    if mean_prior > PRIOR_HIGH:
        return None, 'prior_high'

    out = dict(row)
    out['text'] = cleaned
    out['char_count'] = len(cleaned)          # structural clean changes length; keep in sync
    out['mean_log_prior_stage4'] = mean_prior
    return out, 'ok'


def process_shard4(path_in_repo):
    output_name = os.path.basename(path_in_repo)
    local_source = robust_hf_download(path_in_repo)
    table = pq.read_table(local_source)
    df = table.to_pandas()

    stats = {
        'source_shard': path_in_repo, 'output_shard': output_name,
        'n_input': len(df), 'n_kept': 0, 'n_removed': 0,
        'removed_by_reason': {}, 'chars_in': 0, 'chars_kept': 0,
    }

    kept_rows = []
    removed_by_reason = {}
    for row in df.to_dict('records'):
        stats['chars_in'] += len(row['text']) if isinstance(row.get('text'), str) else 0
        out, reason = process_one_row(row)
        if out is None:
            removed_by_reason[reason] = removed_by_reason.get(reason, 0) + 1
            continue
        kept_rows.append(out)
        stats['chars_kept'] += len(out['text'])

    stats['n_kept'] = len(kept_rows)
    stats['n_removed'] = stats['n_input'] - stats['n_kept']
    stats['removed_by_reason'] = removed_by_reason

    tmp = tempfile.mkdtemp()
    local_out = os.path.join(tmp, output_name)
    local_stats = os.path.join(tmp, f'{Path(output_name).stem}.json')
    out_df = pd.DataFrame(kept_rows) if kept_rows else df.iloc[0:0].copy()
    pq.write_table(pa.Table.from_pandas(out_df, preserve_index=False), local_out)
    with open(local_stats, 'w') as f:
        json.dump(stats, f, indent=2, sort_keys=True)

    api.upload_file(path_or_fileobj=local_out, path_in_repo=f'{DATA4_PREFIX}/{output_name}',
                    repo_id=HF_REPO_ID, repo_type='dataset', commit_message=f'Stage 4 {output_name}')
    api.upload_file(path_or_fileobj=local_stats, path_in_repo=f'{DST4_PREFIX}/stats/{Path(output_name).stem}.json',
                    repo_id=HF_REPO_ID, repo_type='dataset', commit_message=f'Stage 4 stats {output_name}')
    os.remove(local_out); os.remove(local_stats)
    return stats


existing4 = {os.path.basename(f) for f in list_repo_paths()
            if f.startswith(f'{DATA4_PREFIX}/') and f.endswith('.parquet')}
todo4 = [s for s in src4_shards if os.path.basename(s) not in existing4]
print(f"Stage 4 shards: {len(src4_shards)} total | done: {len(src4_shards) - len(todo4)} | remaining: {len(todo4)}")

t0 = time.time()
n_total = len(todo4)
for i, shard in enumerate(todo4, start=1):
    st = process_shard4(shard)
    removed_pct = 100.0 * st['n_removed'] / max(st['n_input'], 1)
    elapsed = time.time() - t0
    avg = elapsed / i
    eta = avg * (n_total - i)
    print(f"[{i}/{n_total}] {st['output_shard']}: kept {st['n_kept']:,}/{st['n_input']:,} "
          f"removed={st['n_removed']:,} ({removed_pct:.2f}%) reasons={st['removed_by_reason']} | "
          f"elapsed {fmt_secs(elapsed)} | ETA ~{fmt_secs(eta)}")

print(f"\nStage 4 run processed {len(todo4)} shard(s) in {fmt_secs(time.time() - t0)}.")


## 4.3 Aggregate report + manifest

Sums every per-shard stats file uploaded so far (safe to run mid-pipeline) and writes
`stage4/manifest.json`.


In [ ]:
stat4_files = sorted(f for f in list_repo_paths() if re.match(rf'^{DST4_PREFIX}/stats/shard_\d+\.json$', f))
print(f"Found {len(stat4_files)} Stage 4 stat file(s).")

totals4 = {'n_shards_done': len(stat4_files), 'n_input': 0, 'n_kept': 0, 'n_removed': 0,
          'chars_in': 0, 'chars_kept': 0, 'removed_by_reason': {}}
for sf in stat4_files:
    local = robust_hf_download(sf)
    with open(local) as f:
        st = json.load(f)
    for k in ['n_input', 'n_kept', 'n_removed', 'chars_in', 'chars_kept']:
        totals4[k] += int(st.get(k, 0))
    for reason, c in st.get('removed_by_reason', {}).items():
        totals4['removed_by_reason'][reason] = totals4['removed_by_reason'].get(reason, 0) + c

totals4['docs_kept_pct'] = 100.0 * totals4['n_kept'] / max(totals4['n_input'], 1)
totals4['docs_removed_pct'] = 100.0 * totals4['n_removed'] / max(totals4['n_input'], 1)
totals4['chars_kept_pct'] = 100.0 * totals4['chars_kept'] / max(totals4['chars_in'], 1)
print(json.dumps(totals4, indent=2, sort_keys=True))

manifest4 = {
    'stage': 4,
    'description': 'Structurally-cleaned + GPT-2 log-prior-filtered copy of stage3 (port of think-dataset-clean)',
    'source_prefix': SRC4_PREFIX,
    'source_shards': len(src4_shards),
    'settings': {
        'min_chars_raw': structural_clean.MIN_CHARS_RAW,
        'min_chars_clean': structural_clean.MIN_CHARS_CLEAN,
        'min_printable': structural_clean.MIN_PRINTABLE,
        'max_ocr_artifacts': structural_clean.MAX_OCR_ARTIFACTS,
        'prior_band': list(PRIOR_BAND),
        'prior_thresholds': PRIOR_THRESHOLDS,
    },
    'totals': totals4,
    'updated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
tmp = tempfile.mkdtemp()
manifest4_path = os.path.join(tmp, 'manifest.json')
with open(manifest4_path, 'w') as f:
    json.dump(manifest4, f, indent=2, sort_keys=True)
api.upload_file(path_or_fileobj=manifest4_path, path_in_repo=f'{DST4_PREFIX}/manifest.json',
                repo_id=HF_REPO_ID, repo_type='dataset', commit_message='Stage 4 manifest')
print("\nUploaded stage4/manifest.json")


---
# Stage 5: Footer Strip + Tiered Anachronism Filter (1930s cutoff)

Port of `think_dataset_clean_1930s.ipynb`. Two sub-stages, run in sequence, reading Stage 4's
output:

- **5a -- footer strip**: line-level removal of reprint/OCR footer lines (URLs, "printed in the
  United States of America", "all rights reserved", photocopy / print-on-demand colophons, ISBN
  lines, bare page numbers, library stamps) via `footer_lib.strip_footers`. Whole books are kept;
  only footer lines go. A document that would lose more than 30% of its lines is kept unstripped
  and flagged, rather than trusting an overzealous match.
- **5b -- anachronism filter**: `build_list`'s tiered 1930-cutoff banned-term list (463 terms:
  tier 1 hard-drops on one hit, tier 2 needs corroboration, tier 3 never fires alone, strip-only
  terms never drop) + `filter_lib`'s fast matcher decide whether to drop each stripped document
  entirely. This is the actual **regex/keyword filtering** stage.


In [ ]:
# === Stage 5 configuration ==================================================
SRC5_PREFIX = 'stage4'                     # input: Stage 4 output
DST5_PREFIX = 'stage5'                     # output root
STRIPPED5_PREFIX = f'{DST5_PREFIX}/stripped'
DATA5_PREFIX = f'{DST5_PREFIX}/data'
BANNED5_PREFIX = f'{DST5_PREFIX}/_banned'

CUTOFF_YEAR = 1930
FORCE_REBUILD_LIST = False   # set True to rebuild the banned list from scratch

# Guardrails against over-stripping (footer_lib module defaults; override if needed).
footer_lib.FOOTER_MAX_LINE_CHARS = 200
footer_lib.FOOTER_MAX_DOC_LINE_FRAC = 0.30

# Scan window + drop policy (filter_lib module defaults; override if needed).
filter_lib.SCAN_CHARS = 300_000
filter_lib.SCAN_TAIL_CHARS = 50_000

print(f"Stage 5: {HF_REPO_ID}/{SRC5_PREFIX} -> {HF_REPO_ID}/{DST5_PREFIX}")
print(f"Cutoff year: {CUTOFF_YEAR} | scan window: head={filter_lib.SCAN_CHARS} tail={filter_lib.SCAN_TAIL_CHARS}")


## 5.0 (optional, guarded) Reset Stage 5 outputs

`WIPE_STRIPPED=True` also clears the footer-stripped layer (forces a full re-strip);
`WIPE_BANNED_LIST=True` also clears the cached banned-term list (forces a rebuild). By default a
reset only removes the final filtered shards/stats/hits/report so you can re-run just the
anachronism filter without re-stripping. Leave `CONFIRM_RESET_STAGE5 = False` for a normal run.


In [ ]:
CONFIRM_RESET_STAGE5 = False
WIPE_STRIPPED = False
WIPE_BANNED_LIST = False

if CONFIRM_RESET_STAGE5:
    stale = []
    for f in list_repo_paths():
        is_stripped = f.startswith(f'{STRIPPED5_PREFIX}/') or f.startswith(f'{DST5_PREFIX}/strip_stats/') \
                     or f.startswith(f'{DST5_PREFIX}/strip_samples/')
        if is_stripped:
            if WIPE_STRIPPED:
                stale.append(f)
            continue
        is_final = (f.startswith(f'{DATA5_PREFIX}/') or f.startswith(f'{DST5_PREFIX}/stats/')
                   or f.startswith(f'{DST5_PREFIX}/hits/') or f == f'{DST5_PREFIX}/manifest.json'
                   or f == f'{DST5_PREFIX}/cleaning_report_1930s.json')
        is_banned = f.startswith(f'{BANNED5_PREFIX}/') if WIPE_BANNED_LIST else False
        if is_final or is_banned:
            stale.append(f)
    print(f"Deleting {len(stale)} file(s) under {DST5_PREFIX}/ ...")
    for f in stale:
        api.delete_file(path_in_repo=f, repo_id=HF_REPO_ID, repo_type='dataset',
                        commit_message=f'Reset Stage 5: remove {f}')
        print(f"  deleted {f}")
    print("Reset complete.")
else:
    print("Reset skipped (CONFIRM_RESET_STAGE5=False). Normal resumable run.")


## 5.1 Build or load the tiered banned-term list

Ports `build_list.build_banned_list()` (pure logic, now in `scripts/build_list.py`): the
croqaz/vintage-ft-v1 seed adjusted from a 1900 to a 1930 cutoff via an allow-list, plus a
post-1930 supplement, tiered by confidence. Cached under `stage5/_banned/` on HF.


In [ ]:
def try_download_banned_list():
    try:
        list_path = robust_hf_download(f'{BANNED5_PREFIX}/banned_list.txt')
        meta_path = robust_hf_download(f'{BANNED5_PREFIX}/list_meta.json')
        tier_path = robust_hf_download(f'{BANNED5_PREFIX}/tiers.json')
        with open(list_path, encoding='utf-8') as f:
            terms = [l.strip().lower() for l in f if l.strip()]
        with open(meta_path, encoding='utf-8') as f:
            meta = json.load(f)
        with open(tier_path, encoding='utf-8') as f:
            raw = json.load(f)
        tiers = {k: (v if v == 'strip' else int(v)) for k, v in raw.items()}
        print("Loaded cached banned list + tiers from HF.")
        return terms, meta, tiers
    except Exception:
        return None, None, None


def build_and_upload_banned_list():
    final, meta, removed_from_seed, added_by_extra, tiers = build_list.build_banned_list()
    tmp = tempfile.mkdtemp()
    d = Path(tmp)
    (d / 'banned_list.txt').write_text('\n'.join(final) + '\n', encoding='utf-8')
    (d / 'allow_list.txt').write_text('\n'.join(sorted(build_list.ALLOW_1930)) + '\n', encoding='utf-8')
    with open(d / 'removed_from_seed.json', 'w', encoding='utf-8') as f:
        json.dump({'removed_from_seed_because_pre_1931': removed_from_seed,
                   'added_by_extra_1930_banned': added_by_extra}, f, indent=2, sort_keys=True)
    with open(d / 'list_meta.json', 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2, sort_keys=True)
    with open(d / 'format_tells.json', 'w', encoding='utf-8') as f:
        json.dump(build_list.FORMAT_TELL_PATTERNS, f, indent=2, sort_keys=True)
    buckets = {1: [], 2: [], 3: [], 'strip': []}
    for term, tier in tiers.items():
        buckets[tier].append(term)
    tier_files = {'tier1.txt': 1, 'tier2.txt': 2, 'tier3.txt': 3, 'strip_only.txt': 'strip'}
    for name, tier in tier_files.items():
        (d / name).write_text('\n'.join(sorted(buckets[tier])) + '\n', encoding='utf-8')
    with open(d / 'tiers.json', 'w', encoding='utf-8') as f:
        json.dump({k: v for k, v in sorted(tiers.items())}, f, indent=2)

    for fname in ['banned_list.txt', 'allow_list.txt', 'removed_from_seed.json', 'list_meta.json',
                 'format_tells.json', 'tier1.txt', 'tier2.txt', 'tier3.txt', 'strip_only.txt', 'tiers.json']:
        api.upload_file(path_or_fileobj=str(d / fname), path_in_repo=f'{BANNED5_PREFIX}/{fname}',
                        repo_id=HF_REPO_ID, repo_type='dataset',
                        commit_message=f'Stage 5 banned list: {fname}')
    print(f"Uploaded banned list ({meta['n_final_terms']} terms; tiers {meta['tier_counts']}).")
    return final, meta, tiers


if FORCE_REBUILD_LIST:
    BANNED_TERMS, LIST_META, TIERS = None, None, None
else:
    BANNED_TERMS, LIST_META, TIERS = try_download_banned_list()
if BANNED_TERMS is None:
    BANNED_TERMS, LIST_META, TIERS = build_and_upload_banned_list()

filter_lib.init_matcher(BANNED_TERMS, TIERS)
tc = LIST_META.get('tier_counts', {})
print(f"Matcher ready: {len(BANNED_TERMS):,} banned terms "
      f"(T1={tc.get('tier1', 0)} T2={tc.get('tier2', 0)} T3={tc.get('tier3', 0)} strip={tc.get('strip', 0)}), "
      f"{len(filter_lib.FORMAT_RES)} format tells. Drop rule: 1x T1 OR >=2 T2/T3 with >=1 T2.")


## 5a. Resumable footer-strip loop

Ports `strip_footers.py`'s `process_shard`. Reads each Stage 4 shard, strips footer/boilerplate
lines from every document's `text` (metadata columns pass through unchanged), and writes to
`stage5/stripped/`. A shard already present there is skipped.


In [ ]:
src5_shards = sorted(
    f for f in list_repo_paths()
    if f.startswith(f'{SRC5_PREFIX}/data/') and f.endswith('.parquet')
)
print(f"Found {len(src5_shards)} Stage 4 shard(s).")
assert src5_shards, "No Stage 4 shards found -- run the Stage 4 cells above first."

SAMPLE_LINES_PER_SHARD = 400


def process_shard_strip(path_in_repo):
    output_name = os.path.basename(path_in_repo)
    stem = Path(output_name).stem
    local_source = robust_hf_download(path_in_repo)
    df = pq.read_table(local_source).to_pandas()

    stats = {
        'source_shard': path_in_repo, 'output_shard': f'{STRIPPED5_PREFIX}/{output_name}',
        'n_docs': len(df), 'n_docs_changed': 0, 'n_docs_flagged_unstripped': 0,
        'n_lines_removed': 0, 'chars_in': 0, 'chars_out': 0, 'removed_by_pattern': {},
    }

    out_rows, samples = [], []
    for doc_idx, row in enumerate(df.to_dict('records')):
        text = row.get('text')
        stats['chars_in'] += len(text) if isinstance(text, str) else 0
        clean, removed, flagged = footer_lib.strip_footers(text)
        if flagged:
            stats['n_docs_flagged_unstripped'] += 1
        for name, line in removed:
            stats['removed_by_pattern'][name] = stats['removed_by_pattern'].get(name, 0) + 1
        if removed and not flagged:
            stats['n_docs_changed'] += 1
            stats['n_lines_removed'] += len(removed)
            if len(samples) < SAMPLE_LINES_PER_SHARD:
                for name, line in removed[:20]:
                    samples.append({'doc_idx': doc_idx, 'pattern': name, 'line': line})
        out_row = dict(row)
        out_row['text'] = clean
        out_rows.append(out_row)
        stats['chars_out'] += len(clean) if isinstance(clean, str) else 0

    stats['chars_removed_pct'] = 100.0 * (stats['chars_in'] - stats['chars_out']) / max(stats['chars_in'], 1)

    tmp = tempfile.mkdtemp()
    local_out = os.path.join(tmp, output_name)
    local_stats = os.path.join(tmp, f'{stem}.json')
    local_samples = os.path.join(tmp, f'{stem}.jsonl')
    out_df = pd.DataFrame(out_rows) if out_rows else df.iloc[0:0].copy()
    pq.write_table(pa.Table.from_pandas(out_df, preserve_index=False), local_out)
    with open(local_stats, 'w') as f:
        json.dump(stats, f, indent=2, sort_keys=True)
    with open(local_samples, 'w') as f:
        for rec in samples:
            f.write(json.dumps(rec) + '\n')

    api.upload_file(path_or_fileobj=local_out, path_in_repo=f'{STRIPPED5_PREFIX}/{output_name}',
                    repo_id=HF_REPO_ID, repo_type='dataset', commit_message=f'Stage 5 strip {output_name}')
    api.upload_file(path_or_fileobj=local_stats, path_in_repo=f'{DST5_PREFIX}/strip_stats/{stem}.json',
                    repo_id=HF_REPO_ID, repo_type='dataset', commit_message=f'Stage 5 strip stats {output_name}')
    api.upload_file(path_or_fileobj=local_samples, path_in_repo=f'{DST5_PREFIX}/strip_samples/{stem}.jsonl',
                    repo_id=HF_REPO_ID, repo_type='dataset', commit_message=f'Stage 5 strip samples {output_name}')
    for p in (local_out, local_stats, local_samples):
        os.remove(p)
    return stats


existing_stripped = {os.path.basename(f) for f in list_repo_paths()
                     if f.startswith(f'{STRIPPED5_PREFIX}/') and f.endswith('.parquet')}
todo_strip = [s for s in src5_shards if os.path.basename(s) not in existing_stripped]
print(f"Footer-strip: {len(src5_shards)} total | already stripped: {len(src5_shards) - len(todo_strip)} | remaining: {len(todo_strip)}")

t0 = time.time()
n_total = len(todo_strip)
for i, shard in enumerate(todo_strip, start=1):
    st = process_shard_strip(shard)
    elapsed = time.time() - t0
    avg = elapsed / i
    eta = avg * (n_total - i)
    top = sorted(st['removed_by_pattern'].items(), key=lambda x: -x[1])[:5]
    print(f"[{i}/{n_total}] {st['output_shard']}: docs {st['n_docs']:,} changed={st['n_docs_changed']:,} "
          f"flagged={st['n_docs_flagged_unstripped']:,} lines_removed={st['n_lines_removed']:,} "
          f"chars-{st['chars_removed_pct']:.2f}% | elapsed {fmt_secs(elapsed)} | ETA ~{fmt_secs(eta)} | top={top}")

print(f"\nFooter-strip processed {len(todo_strip)} shard(s) in {fmt_secs(time.time() - t0)}.")


## 5b. Resumable anachronism-filter loop

Ports `run_filter.py`'s `process_shard`. Reads each footer-stripped shard, scans every document
with the tiered matcher, drops documents on strong evidence of post-1930 content, and writes
survivors (metadata unchanged, `text` unchanged from the stripped layer) to `stage5/data/`. A
shard already present there is skipped.


In [ ]:
def process_shard_filter(path_in_repo):
    output_name = os.path.basename(path_in_repo)
    stem = Path(output_name).stem
    local_source = robust_hf_download(path_in_repo, repo_id=HF_REPO_ID)
    df = pq.read_table(local_source).to_pandas()

    stats = {
        'source_shard': path_in_repo, 'output_shard': output_name,
        'n_input': len(df), 'n_kept': 0, 'n_removed': 0,
        'removed_by_term': {}, 'removed_by_reason': {}, 'boilerplate_hits': {},
        'chars_in': 0, 'chars_kept': 0,
    }

    kept_rows, hit_records = [], []
    for doc_idx, row in enumerate(df.to_dict('records')):
        text = row.get('text')
        stats['chars_in'] += len(text) if isinstance(text, str) else 0
        terms, kinds = filter_lib.scan_text(text)
        drop, reason, by_tier = filter_lib.decide_drop(terms)
        if drop:
            stats['n_removed'] += 1
            stats['removed_by_reason'][reason] = stats['removed_by_reason'].get(reason, 0) + 1
            for t in by_tier[1] + by_tier[2] + by_tier[3]:
                stats['removed_by_term'][t] = stats['removed_by_term'].get(t, 0) + 1
            hit_records.append({'doc_idx': doc_idx, 'reason': reason, 'tier1': by_tier[1],
                                'tier2': by_tier[2], 'tier3': by_tier[3], 'strip': by_tier['strip']})
            continue
        for t in by_tier['strip']:
            stats['boilerplate_hits'][t] = stats['boilerplate_hits'].get(t, 0) + 1
        kept_rows.append(row)   # unchanged -- whole book preserved
        stats['chars_kept'] += len(text) if isinstance(text, str) else 0

    stats['n_kept'] = len(kept_rows)

    tmp = tempfile.mkdtemp()
    local_out = os.path.join(tmp, output_name)
    local_stats = os.path.join(tmp, f'{stem}.json')
    local_hits = os.path.join(tmp, f'{stem}.jsonl')
    out_df = pd.DataFrame(kept_rows) if kept_rows else df.iloc[0:0].copy()
    pq.write_table(pa.Table.from_pandas(out_df, preserve_index=False), local_out)
    with open(local_stats, 'w') as f:
        json.dump(stats, f, indent=2, sort_keys=True)
    with open(local_hits, 'w') as f:
        for rec in hit_records:
            f.write(json.dumps(rec) + '\n')

    api.upload_file(path_or_fileobj=local_out, path_in_repo=f'{DATA5_PREFIX}/{output_name}',
                    repo_id=HF_REPO_ID, repo_type='dataset', commit_message=f'Stage 5 filter {output_name}')
    api.upload_file(path_or_fileobj=local_stats, path_in_repo=f'{DST5_PREFIX}/stats/{stem}.json',
                    repo_id=HF_REPO_ID, repo_type='dataset', commit_message=f'Stage 5 filter stats {output_name}')
    api.upload_file(path_or_fileobj=local_hits, path_in_repo=f'{DST5_PREFIX}/hits/{stem}.jsonl',
                    repo_id=HF_REPO_ID, repo_type='dataset', commit_message=f'Stage 5 filter hits {output_name}')
    for p in (local_out, local_stats, local_hits):
        os.remove(p)
    return stats


all_stripped = sorted(f for f in list_repo_paths()
                      if f.startswith(f'{STRIPPED5_PREFIX}/') and f.endswith('.parquet'))
existing_filtered = {os.path.basename(f) for f in list_repo_paths()
                     if f.startswith(f'{DATA5_PREFIX}/') and f.endswith('.parquet')}
todo_filter = [s for s in all_stripped if os.path.basename(s) not in existing_filtered]
print(f"Anachronism filter: {len(all_stripped)} stripped shard(s) total | "
      f"already filtered: {len(all_stripped) - len(todo_filter)} | remaining: {len(todo_filter)}")

t0 = time.time()
n_total = len(todo_filter)
for i, shard in enumerate(todo_filter, start=1):
    st = process_shard_filter(shard)
    removed_pct = 100.0 * st['n_removed'] / max(st['n_input'], 1)
    elapsed = time.time() - t0
    avg = elapsed / i
    eta = avg * (n_total - i)
    top = sorted(st['removed_by_term'].items(), key=lambda x: -x[1])[:5]
    print(f"[{i}/{n_total}] {st['output_shard']}: kept {st['n_kept']:,}/{st['n_input']:,} "
          f"removed={st['n_removed']:,} ({removed_pct:.2f}%) | elapsed {fmt_secs(elapsed)} | "
          f"ETA ~{fmt_secs(eta)} | top={top}")

print(f"\nAnachronism filter processed {len(todo_filter)} shard(s) in {fmt_secs(time.time() - t0)}.")


## 5.2 Aggregate report + manifest

Sums footer-strip and filter stats across every completed shard (safe to run mid-pipeline) and
writes `stage5/cleaning_report_1930s.json` + `stage5/manifest.json`.


In [ ]:
strip_stat_files = sorted(f for f in list_repo_paths() if re.match(rf'^{DST5_PREFIX}/strip_stats/shard_\d+\.json$', f))
filter_stat_files = sorted(f for f in list_repo_paths() if re.match(rf'^{DST5_PREFIX}/stats/shard_\d+\.json$', f))
print(f"Found {len(strip_stat_files)} strip stat file(s) and {len(filter_stat_files)} filter stat file(s).")

from collections import Counter

strip_totals = {'n_shards_stripped': len(strip_stat_files), 'n_docs': 0, 'n_docs_changed': 0,
                'n_docs_flagged_unstripped': 0, 'n_lines_removed': 0, 'chars_in': 0, 'chars_out': 0,
                'removed_by_pattern': Counter()}
for sf in strip_stat_files:
    local = robust_hf_download(sf)
    with open(local) as f:
        st = json.load(f)
    for k in ['n_docs', 'n_docs_changed', 'n_docs_flagged_unstripped', 'n_lines_removed', 'chars_in', 'chars_out']:
        strip_totals[k] += int(st.get(k, 0))
    strip_totals['removed_by_pattern'].update(st.get('removed_by_pattern', {}))
top_footers = strip_totals['removed_by_pattern'].most_common(20)
strip_totals['top_footer_patterns'] = top_footers
strip_totals['removed_by_pattern'] = dict(strip_totals['removed_by_pattern'])
strip_totals['chars_removed_pct'] = 100.0 * (strip_totals['chars_in'] - strip_totals['chars_out']) / max(strip_totals['chars_in'], 1)

totals5 = {
    'cutoff_year': CUTOFF_YEAR, 'banned_list_meta': LIST_META, 'n_banned_terms': len(BANNED_TERMS),
    'n_shards_done': len(filter_stat_files), 'tier_counts': LIST_META.get('tier_counts', {}),
    'n_input': 0, 'n_kept': 0, 'n_removed': 0, 'chars_in': 0, 'chars_kept': 0,
    'removed_by_term': Counter(), 'removed_by_reason': Counter(), 'boilerplate_hits': Counter(),
}
for sf in filter_stat_files:
    local = robust_hf_download(sf)
    with open(local) as f:
        st = json.load(f)
    for k in ['n_input', 'n_kept', 'n_removed', 'chars_in', 'chars_kept']:
        totals5[k] += int(st.get(k, 0))
    totals5['removed_by_term'].update(st.get('removed_by_term', {}))
    totals5['removed_by_reason'].update(st.get('removed_by_reason', {}))
    totals5['boilerplate_hits'].update(st.get('boilerplate_hits', {}))

top_terms = totals5['removed_by_term'].most_common(30)
totals5['top_firing_terms'] = top_terms
totals5['removed_by_term'] = dict(totals5['removed_by_term'])
totals5['removed_by_reason'] = dict(totals5['removed_by_reason'])
totals5['boilerplate_hits'] = dict(totals5['boilerplate_hits'])
totals5['docs_kept_pct'] = 100.0 * totals5['n_kept'] / max(totals5['n_input'], 1)
totals5['docs_removed_pct'] = 100.0 * totals5['n_removed'] / max(totals5['n_input'], 1)
totals5['chars_kept_pct'] = 100.0 * totals5['chars_kept'] / max(totals5['chars_in'], 1)
totals5['footer_strip'] = strip_totals
totals5['updated_at'] = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

print(json.dumps({k: v for k, v in totals5.items() if k != 'removed_by_term'}, indent=2, sort_keys=True))

tmp = tempfile.mkdtemp()
report5_path = os.path.join(tmp, 'cleaning_report_1930s.json')
with open(report5_path, 'w') as f:
    json.dump(totals5, f, indent=2, sort_keys=True)
api.upload_file(path_or_fileobj=report5_path, path_in_repo=f'{DST5_PREFIX}/cleaning_report_1930s.json',
                repo_id=HF_REPO_ID, repo_type='dataset', commit_message='Stage 5 cleaning report')

manifest5 = {
    'stage': 5,
    'description': 'Footer-stripped + tiered-anachronism-filtered copy of stage4 (port of think-dataset-clean-1930s)',
    'source_prefix': SRC5_PREFIX,
    'cutoff_year': CUTOFF_YEAR,
    'n_banned_terms': len(BANNED_TERMS),
    'tier_counts': LIST_META.get('tier_counts', {}),
    'totals': totals5,
}
manifest5_path = os.path.join(tmp, 'manifest.json')
with open(manifest5_path, 'w') as f:
    json.dump(manifest5, f, indent=2, sort_keys=True)
api.upload_file(path_or_fileobj=manifest5_path, path_in_repo=f'{DST5_PREFIX}/manifest.json',
                repo_id=HF_REPO_ID, repo_type='dataset', commit_message='Stage 5 manifest')
print(f"\nUploaded stage5/cleaning_report_1930s.json + stage5/manifest.json. "
      f"Removed so far: {totals5['n_removed']:,} ({totals5['docs_removed_pct']:.2f}%).")


## 5.3 Read-back verification

Confirms the uploaded Stage 5 corpus is intact.


In [ ]:
check_manifest5 = robust_hf_download(f'{DST5_PREFIX}/manifest.json')
with open(check_manifest5) as f:
    print(json.dumps(json.load(f), indent=2)[:2000])

final5_shards = sorted(f for f in list_repo_paths() if f.startswith(f'{DATA5_PREFIX}/') and f.endswith('.parquet'))
if final5_shards:
    first5 = robust_hf_download(final5_shards[0])
    check_df5 = pq.read_table(first5).to_pandas()
    print(f"\nFirst Stage 5 shard: {len(check_df5):,} docs")
    cols = [c for c in ['title_src', 'ocroscope_ratio_segment', 'char_count'] if c in check_df5.columns]
    display(check_df5[cols].head())
    print("\nStage 5 (anachronism filter) complete.")
else:
    print("No Stage 5 output shards yet.")


---
# README: Upload the Pipeline Description to HF

Writes a top-level `README.md` to `zachnorton03/think-midtrain` describing every stage of the
midtraining pipeline end to end (Stage 1 through Stage 5), so the dataset repo is self-documenting
for anyone browsing it on HF. Safe to re-run at any point -- it reflects whatever manifests exist
on HF at the time it runs.


In [ ]:
def _try_load_json(path_in_repo):
    try:
        local = robust_hf_download(path_in_repo)
        with open(local) as f:
            return json.load(f)
    except Exception:
        return None

_m2 = _try_load_json('stage2/manifest.json')
_m3 = _try_load_json('stage3/manifest.json')
_m4 = _try_load_json(f'{DST4_PREFIX}/manifest.json')
_m5 = _try_load_json(f'{DST5_PREFIX}/manifest.json')

def _fnum(x):
    return f'{x:,}' if isinstance(x, (int, float)) else 'n/a'

s2_docs = _fnum(_m2['totals']['documents']) if _m2 else 'n/a'
s2_tokens = _fnum(_m2['totals']['tokens_o200k_base']) if _m2 else 'n/a'
s3_in = _fnum(_m3['input_docs']) if _m3 else 'n/a'
s3_out = _fnum(_m3['output_docs']) if _m3 else 'n/a'
s4_in = _fnum(_m4['totals']['n_input']) if _m4 else 'n/a'
s4_out = _fnum(_m4['totals']['n_kept']) if _m4 else 'n/a'
s5_in = _fnum(_m5['totals']['n_input']) if _m5 else 'n/a'
s5_out = _fnum(_m5['totals']['n_kept']) if _m5 else 'n/a'

readme = f'''---
dataset_info:
  features:
  - name: text
    dtype: string
---

# think-midtrain

Midtraining corpus assembled from pre-1930s public-domain STEM-leaning text, built in five
sequential stages. Each stage reads the previous stage's output from this repo and writes its own
prefix; every stage keeps whole documents as single rows and preserves per-document metadata
(title, OCR scores, token counts, source identifiers, ...) alongside `text`.

## Pipeline stages

| Stage | Prefix | Notebook | What it does |
|---|---|---|---|
| 1 | (external: `jbduran/think-dataset`) | `midtraining_extraction.ipynb` | LC-classification + OCR/tokenizability filtering of `jbduran/think-dataset`, targeting underrepresented Science/Technology/Medicine subjects. |
| 2 | `stage2/` | `midtraining_extraction.ipynb` | Internet Archive harvest (math + STEM collections, pre-1930 date range), heuristic OCR-confidence + language + dedup filtering. |
| 3 | `stage3/` | `midtraining_ocroscope_filtering.ipynb` | Second-pass OCR quality filter using Pleias/OCRoscope (7-gram language-consistency + non-character ratio), independent of Stage 2's dictionary-hit heuristic. |
| 4 | `stage4/` | `midtraining_stage4_cleaning.ipynb` | Boilerplate/OCR cleanup (Google Books, HathiTrust, library stamps, Project Gutenberg headers, hyphenation, front matter) + a GPT-2 token log-prior filter (p2.5-p97.5) pruning statistical outliers. Port of `think-dataset-clean`. |
| 5 | `stage5/` | `midtraining_stage4_cleaning.ipynb` | Footer/boilerplate line stripping, then a tiered 1930-cutoff anachronism keyword filter (regex/keyword matcher; a document is dropped only on strong evidence of post-1930 content). Port of `think-dataset-clean-1930s`. |

## Current totals (as of the last manifest update on each prefix)

- Stage 2 (Internet Archive harvest): {s2_docs} documents, {s2_tokens} tokens (o200k_base)
- Stage 3 (OCRoscope filter): {s3_in} in -> {s3_out} kept
- Stage 4 (structural clean + log-prior filter): {s4_in} in -> {s4_out} kept
- Stage 5 (footer strip + anachronism filter): {s5_in} in -> {s5_out} kept

## Resumability

Every stage is resumable at shard granularity: a rerun skips any output shard already present on
HF and continues with whatever remains. Stage 4's GPT-2 log-prior table and Stage 5's banned-term
list are each built once, cached under `stage4/_prior/` and `stage5/_banned/`, and reused on
every subsequent run instead of being recomputed.

## Artifacts per stage

- `stage2/`: `data/`, `audit_metadata.jsonl`, `checkpoint.jsonl`, `manifest.json`
- `stage3/`: `data/`, `scored/` (OCRoscope score checkpoints), `manifest.json`
- `stage4/`: `data/`, `stats/`, `_prior/` (cached log-prior table + thresholds), `manifest.json`
- `stage5/`: `stripped/`, `strip_stats/`, `strip_samples/`, `data/`, `stats/`, `hits/`, `_banned/` (cached term list + tiers), `cleaning_report_1930s.json`, `manifest.json`

Updated: {time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
'''

tmp = tempfile.mkdtemp()
readme_path = os.path.join(tmp, 'README.md')
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme)
api.upload_file(path_or_fileobj=readme_path, path_in_repo='README.md',
                repo_id=HF_REPO_ID, repo_type='dataset', commit_message='Update pipeline README')
print(readme)
print(f"\nUploaded README.md to {HF_REPO_ID}")
